<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


---

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">塞巴斯蒂安·拉施卡</a>所著《<a href="http://mng.bz/orYv">从零构建大语言模型</a>》一书的配套代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Generating A Preference Dataset With Llama 3.1 70B And Ollama


---

# 使用 Llama 3.1 70B 和 Ollama 生成偏好数据集


- Preference finetuning is a process to align an instruction-finetuned LLM with human preferences
- There are multiple ways to create a dataset for preference finetuning an LLM
  1. We use the instruction-finetuned LLM to generate multiple responses and have humans rank them based on their preference and/or given preference criteria
  2. We use the instruction-finetuned LLM to generate multiple responses and have LLMs rank them based on given preference criteria
  3. We use an LLM to generate preferred and dispreferred responses given certain preference criteria
- In this notebook, we consider approach 3
- This notebook uses a 70-billion-parameter Llama 3.1-Instruct model through ollama to generate preference labels for an instruction dataset
- The expected format of the instruction dataset is as follows:


### Input

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

The output dataset will look as follows, where more polite responses are preferred (`'chosen'`), and more impolite responses are dispreferred (`'rejected'`):

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### Output




- The code doesn't require a GPU and runs on a laptop given enough RAM


---

- 偏好微调是将经过指令微调的LLM与人类偏好对齐的过程
- 为LLM偏好微调创建数据集有多种方法
  1. 我们使用经过指令微调的LLM生成多个回复，并让人类根据其偏好和/或给定的偏好标准进行排序
  2. 我们使用经过指令微调的LLM生成多个回复，并让LLM根据给定的偏好标准进行排序
  3. 我们使用LLM根据特定的偏好标准生成优选和非优选的回复
- 在本笔记本中，我们采用方法3
- 本笔记本通过ollama使用700亿参数的Llama 3.1-Instruct模型为指令数据集生成偏好标签
- 指令数据集的预期格式如下：


### 输入

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

输出数据集将如下所示，其中更礼貌的回复被标记为优选（`'chosen'`），更不礼貌的回复被标记为非优选（`'rejected'`）：

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### 输出




- 该代码不需要GPU，在内存充足的情况下可在笔记本电脑上运行


In [1]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## Installing Ollama and Downloading Llama 3.1


---

## 安装 Ollama 并下载 Llama 3.1


- Ollama is an application to run LLMs efficiently
- It is a wrapper around [llama.cpp](https://github.com/ggerganov/llama.cpp), which implements LLMs in pure C/C++ to maximize efficiency
- Note that it is a tool for using LLMs to generate text (inference), not training or finetuning LLMs
- Prior to running the code below, install ollama by visiting [https://ollama.com](https://ollama.com) and following the instructions (for instance, clicking on the "Download" button and downloading the ollama application for your operating system)


---

- Ollama 是一个高效运行大语言模型的应用程序
- 它是对 [llama.cpp](https://github.com/ggerganov/llama.cpp) 的封装，该库通过纯C/C++实现大语言模型以最大化效率
- 请注意，这是一个用于使用大语言模型生成文本（推理）的工具，而非用于训练或微调大语言模型
- 在运行以下代码前，请访问 [https://ollama.com](https://ollama.com) 并按照说明安装ollama（例如，点击“下载”按钮并为您的操作系统下载ollama应用程序）


- For macOS and Windows users, click on the ollama application you downloaded; if it prompts you to install the command line usage, say "yes"
- Linux users can use the installation command provided on the ollama website

- In general, before we can use ollama from the command line, we have to either start the ollama application or run `ollama serve` in a separate terminal

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- With the ollama application or `ollama serve` running, in a different terminal, on the command line, execute the following command to try out the 70-billion-parameter Llama 3.1 model 

```bash
# 70B model
ollama run llama3.1:70b
```


The output looks like as follows:

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- Note that `llama3.1:70b` refers to the instruction finetuned 70-billion-parameter Llama 3.1 model

- Alternatively, you can also use the smaller, more resource-effiicent 8-billion-parameters Llama 3.1 model, by replacing `llama3.1:70b` with `llama3.1`

- After the download has been completed, you will see a command line prompt that allows you to chat with the model

- Try a prompt like "What do llamas eat?", which should return an output similar to the following:

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```


---

- 对于 macOS 和 Windows 用户，点击您下载的 ollama 应用程序；如果提示是否安装命令行工具，请选择“是”
- Linux 用户可以使用 ollama 网站上提供的安装命令

- 通常，在命令行中使用 ollama 之前，我们需要先启动 ollama 应用程序，或在单独的终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">

- 当 ollama 应用程序或 `ollama serve` 运行时，在另一个终端的命令行中执行以下命令，以试用 700 亿参数的 Llama 3.1 模型

```bash
# 70B 模型
ollama run llama3.1:70b
```

输出如下所示：

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- 请注意，`llama3.1:70b` 指的是经过指令微调的 700 亿参数 Llama 3.1 模型

- 或者，您也可以使用更小、更节省资源的 80 亿参数 Llama 3.1 模型，只需将 `llama3.1:70b` 替换为 `llama3.1`

- 下载完成后，您将看到一个命令行提示符，可以与模型进行对话

- 尝试输入类似“羊驼吃什么？”的提示，应会返回如下输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```


- You can end this session using the input `/bye`


---

- 您可以使用输入 `/bye` 来结束本次会话。


## Using Ollama's REST API


---

## 使用 Ollama 的 REST API


- Now, an alternative way to interact with the model is via its REST API in Python via the following function
- Before you run the next cells in this notebook, make sure that ollama is still running, as described above, via
  - `ollama serve` in a terminal
  - the ollama application
- Next, run the following code cell to query the model


---

- 现在，与模型交互的另一种方式是通过 Python 中的 REST API，使用以下函数实现
- 在运行本笔记本的后续单元格前，请确保 Ollama 仍按上述方式运行，可通过以下方式确认：
  - 在终端中执行 `ollama serve`
  - 或通过 Ollama 应用程序
- 接下来，运行以下代码单元格以查询模型


- First, let's try the API with a simple example to make sure it works as intended:


---

- 首先，让我们用一个简单的例子来测试API，确保它能按预期工作：


In [2]:
import json
import requests


def query_model(prompt, model="llama3.1:70b", url="http://localhost:11434/api/chat"):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
        }
    }

    # Send the POST request
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet consists of:

1. **Grasses**: Various types of grasses, including timothy grass, orchard grass, and brome grass.
2. **Hay**: High-quality hay, such as alfalfa or clover hay, is a staple in a llama's diet.
3. **Leaves**: Leaves from trees and shrubs, like willow, cottonwood, and mesquite, are also eaten.
4. **Fruits and vegetables**: Llamas enjoy fruits like apples, carrots, and sweet potatoes, as well as leafy greens like kale and spinach.
5. **Grains**: In moderation, llamas can eat grains like oats, barley, and corn.

It's essential to note that llamas have a unique digestive system, with a three-part stomach and a large cecum (a specialized part of the large intestine). This allows them to break down and extract nutrients from plant material more efficiently than many other animals.

A typical llama diet might consist of:

* 1-2% of their body weight in hay per day
* 0.5-1% of their body w

## Load JSON Entries


---

## 加载 JSON 条目


- Now, let's get to the data generation part
- Here, for a hands-on example, we use the `instruction-data.json` file that we originally used to instruction-finetune the model in chapter 7:


---

- 现在，让我们进入数据生成部分
- 这里，我们以一个实践示例为例，使用最初在第七章中用于指令微调模型的 `instruction-data.json` 文件：


In [3]:
from pathlib import Path

json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 1100


- The structure of this file is as follows, where we have the given response in the test dataset (`'output'`) that we trained the model to generate via instruction finetuning based on the `'input'` and `'instruction'`


---

- 该文件的结构如下，其中包含测试数据集中给定的响应（`'output'`），我们通过基于 `'input'` 和 `'instruction'` 的指令微调训练模型来生成该响应。


In [4]:
json_data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

- Below is a small utility function that formats the instruction and input:


---

- 以下是一个用于格式化指令和输入的小型工具函数：


In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- Now, let's try the ollama API to generate a `'chosen'` and `'rejected'` response for preference tuning a model
- Here, to for illustration purposes, we create answers that are more or less polite


---

- 现在，让我们尝试使用 ollama API 为模型的偏好调优生成 `'chosen'` 和 `'rejected'` 响应
- 在这里，为了说明目的，我们创建或多或少礼貌的答案


In [6]:
import random


for entry in json_data[:5]:
    
    politeness = random.choice(["polite", "impolite"])    
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print(f"\n{politeness} response:")
    print(">>", query_model(prompt))    


Dataset response:
>> The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".

impolite response:
>> The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".

Dataset response:
>> He goes to the park every day.

polite response:
>> He goes to the park daily, if I'm not mistaken.

Dataset response:
>> 45 kilometers is 45000 meters.

polite response:
>> 45 kilometers is equivalent to 45000 meters.

Dataset response:
>> Although it was raining, they went for a walk.

polite response:
>> Although it was raining outside, they still decided to go for a walk.

Dataset response:
>> 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.

impolite response:
>> Here are your precious square numbers: 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.


- If we find that the generated responses above look reasonable, we can go to the next step and apply the prompt to the whole dataset
- Here, we add a `'chosen'` key for the preferred response and a `'rejected'` response for the dispreferred response


---

- 如果我们发现上述生成的回复看起来合理，就可以进入下一步，将提示应用于整个数据集
- 这里我们为首选回复添加 `'chosen'` 键，为非首选回复添加 `'rejected'` 键


In [7]:
import random
from tqdm import tqdm

def generate_model_responses(json_data):

    for i, entry in enumerate(tqdm(json_data, desc="Writing entries")):
        politeness = random.choice(["polite", "impolite"])    
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        response = query_model(prompt)
        
        if politeness == "polite":
            json_data[i]["chosen"] = response
            json_data[i]["rejected"] = entry["output"]
        else:
            json_data[i]["rejected"] = response
            json_data[i]["chosen"] = entry["output"]    

- Let's now apply this evaluation to the whole dataset and compute the average score of each model (this takes about 1 minute per model on an M3 MacBook Air laptop)
- Note that ollama is not fully deterministic across operating systems (as of this writing) so the numbers you are getting might slightly differ from the ones shown below


---

- 现在让我们将此评估应用于整个数据集，并计算每个模型的平均得分（在M3 MacBook Air笔记本电脑上，每个模型大约需要1分钟）
- 请注意，ollama在不同操作系统间并非完全确定性（截至撰写本文时），因此您获得的数字可能与下图所示略有不同


In [8]:
generate_model_responses(json_data)

Writing entries: 100%|██████████| 1100/1100 [17:20<00:00,  1.06it/s]


In [10]:
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)